# Week 7 — Content Action Playbook (ML-10)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoyuQre/flyrank_assgn_1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the **validated** Week-5/6 Random Forest into a practical, human-reviewed **content action playbook**. The chain the notebook demonstrates:

Validated model → predicted decline → business/context signals → priority ranking → deterministic reason code → content archetype → recommended action → human review → monitoring / retrain policy.

Everything is recomputed from the dataset using the **same pipeline as the Week-6 validation audit** (`w06_validation_audit.ipynb`): same target, same excluded columns, same encodings, same Random Forest (`n_estimators=200`, `random_state=42`). The model is trained on the **grouped-by-client train fold** and the queue is built **only from the held-out test fold** (7 clients, 6,163 rows), so every probability in the queue is out-of-sample. No training-fold row is scored.

The playbook's core position: **model prediction ≠ automatic action.** The model prioritizes and flags; a human validates the recommendation and decides what, if anything, to change.

Language note: this notebook uses *observed, measured, directional, decision-support* language. It does not claim that refreshing a page causes an outcome, and it does not promise revenue.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**What the queue is.** A ranked list of the 6,163 held-out content items, best first. A reviewer reads the top of the list, checks the page, and decides whether to refresh or leave it.

**Priority score (a labeled heuristic, not revenue).**

```
priority_score = 100 × (0.6 × decline_probability + 0.3 × impression value + 0.1 × staleness)
```

* decline probability — the model's out-of-sample `P(declining)` on the held-out fold.
* impression value — percentile rank of `impressions_90d` within the scored set (business exposure proxy).
* staleness — `days_since_last_update / 365`, capped at 1.

The formula, the components, and the normalization are documented above. This is a **prioritization heuristic** — this dataset has no revenue data, so the score does not estimate money.

**Priority tiers (proposed decision-policy thresholds, set from the actual score distribution):**

| Tier | Priority score | Meaning |
| --- | ---: | --- |
| `P1_HIGH` | ≥ 70 | Review first |
| `P2_MEDIUM` | 55–70 | Review soon |
| `P3_MONITOR` | 40–55 | Monitor |
| `P4_LOW` | < 40 | Routine only |

**Deterministic reason-code families** (rule-based from observed/model-derived signals — no LLM explanations):

| Code | Fires when | Archetype | Recommended action |
| --- | --- | --- | --- |
| `RC01_HIGH_VALUE_DECLINE` | predicted decline and `impressions_90d` ≥ 75th pct | A — High-value declining | Prioritize human content-refresh review |
| `RC02_CTR_OPPORTUNITY` | `impressions_90d` ≥ 50th pct and `ctr` < 0.5% | B — High-impression / low-CTR | Review title, meta description, SERP alignment, search intent |
| `RC03_AGING_CONTENT` | `days_since_last_update` ≥ 90 and (predicted decline or ≥ 50th-pct impressions) | C — Aging content | Freshness and factual-content review |
| `RC04_DEEP_POSITION_RISK` | predicted decline and `avg_position > 10` (0 = no data, never a rank) | D — Ranking deterioration | Investigate relevance, quality, linking, competition |
| `RC05_MODEL_ONLY_WARNING` | predicted decline, no stronger observable signal matched | E — Model-only warning | Manual investigation before any action |
| `RC06_MONITOR` | probability 0.3–0.5 | F — Monitor | Re-check next cycle |
| `RC07_LOW_PRIORITY` | probability < 0.3 | G — Low priority | Routine monitoring only |

`ctr` is a ×100 rate column in this dataset, so `ctr < 0.5` means below 0.5%. A row can match several families; the **primary** reason code is the first match in the order above, and every matched family is kept in `reason_codes` with an `evidence_count`.

Section 1 code: load the data, reproduce the validated pipeline, fit the model on the grouped train fold, score the held-out fold, assign reason codes / archetypes / actions, rank the queue, and show the priority-tier and reason-code distributions.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

def find_csv():
    for path in [
        "data/raw/content_refresh_anonymized.csv",
        "content_refresh_anonymized.csv",
        "/content/content_refresh_anonymized.csv",
    ]:
        if os.path.exists(path):
            return path
    current = os.path.abspath(os.getcwd())
    for _ in range(6):
        probe = os.path.join(current, "data", "raw", "content_refresh_anonymized.csv")
        if os.path.exists(probe):
            return probe
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise FileNotFoundError("Could not locate content_refresh_anonymized.csv")

def repo_root():
    # csv_path is <repo>/data/raw/content_refresh_anonymized.csv -> walk up 3 levels
    csv_path = find_csv()
    return os.path.dirname(os.path.dirname(os.path.dirname(csv_path)))

df = pd.read_csv(find_csv())

# --- Week-5/6 pipeline reproduced verbatim from w06_validation_audit.ipynb ---
y = (df["trend_direction"] == "down").astype(int)
drop_cols = ["content_id", "client_id", "trend_direction", "trend_pct"]
X = df.drop(columns=drop_cols)
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns
X[num_cols] = X[num_cols].fillna(0)
X[cat_cols] = X[cat_cols].fillna("Unknown")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Honest grouped-by-client split, identical to Week 6
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
assert len(train_clients & test_clients) == 0, "Client leakage between folds!"

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
prob = rf.predict_proba(X_test)[:, 1]
pred = rf.predict(X_test)

print("Train fold :", len(train_idx), "rows /", len(train_clients), "clients")
print("Test fold  :", len(test_idx), "rows /", len(test_clients), "clients")
print("Client overlap:", len(train_clients & test_clients))
print("Test base rate ('down' label):", round(float(y_test.mean()), 4))

# --- Ranked action queue on the held-out fold only ---
queue = df.iloc[test_idx].copy()
queue["decline_probability"] = np.round(prob, 4)
queue["predicted_declining"] = prob >= 0.5
queue["y_true"] = y_test.values
queue["y_pred"] = pred

p50_imp = queue["impressions_90d"].quantile(0.50)
p75_imp = queue["impressions_90d"].quantile(0.75)

# Deterministic, transparent signal flags
queue["flag_high_value"] = queue["impressions_90d"] >= p75_imp
queue["flag_good_volume"] = queue["impressions_90d"] >= p50_imp
queue["flag_low_ctr"] = queue["ctr"] < 0.5          # ctr is a x100 rate: <0.5 means <0.5%
queue["flag_aging"] = queue["days_since_last_update"] >= 90
queue["flag_deep_position"] = (queue["avg_position"] > 0) & (queue["avg_position"] > 10)
queue["flag_high_conf"] = queue["decline_probability"] >= 0.8

def reason_codes(row):
    # Return every reason-code family that matches (deterministic rules).
    codes = []
    if row["predicted_declining"] and row["flag_high_value"]:
        codes.append("RC01_HIGH_VALUE_DECLINE")
    if row["flag_good_volume"] and row["flag_low_ctr"]:
        codes.append("RC02_CTR_OPPORTUNITY")
    if row["flag_aging"] and (row["predicted_declining"] or row["flag_good_volume"]):
        codes.append("RC03_AGING_CONTENT")
    if row["predicted_declining"] and row["flag_deep_position"]:
        codes.append("RC04_DEEP_POSITION_RISK")
    if row["predicted_declining"] and not codes:
        codes.append("RC05_MODEL_ONLY_WARNING")
    return codes

def primary_code(codes, row):
    for code in [
        "RC01_HIGH_VALUE_DECLINE", "RC02_CTR_OPPORTUNITY", "RC03_AGING_CONTENT",
        "RC04_DEEP_POSITION_RISK", "RC05_MODEL_ONLY_WARNING",
    ]:
        if code in codes:
            return code
    if 0.3 <= row["decline_probability"] < 0.5:
        return "RC06_MONITOR"
    return "RC07_LOW_PRIORITY"

REASON_TEXT = {
    "RC01_HIGH_VALUE_DECLINE": "High-traffic page (impressions_90d at or above the 75th percentile of the scored set) that the model predicts is declining; highest-value refresh-review candidate.",
    "RC02_CTR_OPPORTUNITY": "Above-median impressions with a click-through rate below 0.5%; candidate to review title, meta description, and search-intent alignment (this alone is not evidence of decline).",
    "RC03_AGING_CONTENT": "Content not updated in 90+ days, with either model-predicted decline or above-median traffic; candidate for a freshness and factual-accuracy review.",
    "RC04_DEEP_POSITION_RISK": "Model-predicted decline on a page currently positioned beyond page 1 (avg_position > 10; a value of 0 means no position data and is never treated as a rank); candidate to investigate relevance, quality, and competition.",
    "RC05_MODEL_ONLY_WARNING": "Model predicts decline but no strong observable supporting signal matched in this rule set; manual investigation is required before any action.",
    "RC06_MONITOR": "Moderate model risk (probability 0.3-0.5); no action now, re-check at the next review cycle.",
    "RC07_LOW_PRIORITY": "Low model risk (probability below 0.3); routine monitoring only.",
}
ARCHETYPE = {
    "RC01_HIGH_VALUE_DECLINE": "A_HIGH_VALUE_DECLINE",
    "RC02_CTR_OPPORTUNITY": "B_CTR_OPPORTUNITY",
    "RC03_AGING_CONTENT": "C_AGING_CONTENT",
    "RC04_DEEP_POSITION_RISK": "D_DEEP_POSITION_RISK",
    "RC05_MODEL_ONLY_WARNING": "E_MODEL_ONLY_WARNING",
    "RC06_MONITOR": "F_MONITOR",
    "RC07_LOW_PRIORITY": "G_LOW_PRIORITY",
}
RECOMMENDED_ACTION = {
    "RC01_HIGH_VALUE_DECLINE": "Prioritize human content-refresh review.",
    "RC02_CTR_OPPORTUNITY": "Review title, meta description, SERP alignment, and search intent.",
    "RC03_AGING_CONTENT": "Perform a freshness and factual-content review.",
    "RC04_DEEP_POSITION_RISK": "Investigate relevance, content quality, internal linking, and competitive changes.",
    "RC05_MODEL_ONLY_WARNING": "Manual investigation before taking any action.",
    "RC06_MONITOR": "Monitor; review again at the next cycle if signals persist.",
    "RC07_LOW_PRIORITY": "No immediate action; include in routine monitoring.",
}

queue["reason_codes"] = queue.apply(reason_codes, axis=1)
queue["reason_code"] = queue.apply(lambda r: primary_code(r["reason_codes"], r), axis=1)
queue["reason_text"] = queue["reason_code"].map(REASON_TEXT)
queue["archetype"] = queue["reason_code"].map(ARCHETYPE)
queue["recommended_action"] = queue["reason_code"].map(RECOMMENDED_ACTION)
queue["evidence_count"] = queue["reason_codes"].apply(len)
queue["human_review_required"] = queue["reason_code"].isin([
    "RC01_HIGH_VALUE_DECLINE", "RC02_CTR_OPPORTUNITY", "RC03_AGING_CONTENT",
    "RC04_DEEP_POSITION_RISK", "RC05_MODEL_ONLY_WARNING",
])

# Priority score: labeled heuristic (0.6 risk + 0.3 value + 0.1 staleness), NOT revenue
imp_pct = queue["impressions_90d"].rank(pct=True)
stale_score = (queue["days_since_last_update"] / 365.0).clip(upper=1.0)
queue["priority_score"] = (100 * (0.6 * queue["decline_probability"] + 0.3 * imp_pct + 0.1 * stale_score)).round(1)

def priority_tier(score):
    if score >= 70:
        return "P1_HIGH"
    if score >= 55:
        return "P2_MEDIUM"
    if score >= 40:
        return "P3_MONITOR"
    return "P4_LOW"

queue["priority_tier"] = queue["priority_score"].apply(priority_tier)
queue = queue.sort_values(["priority_score", "decline_probability"], ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print()
print("Queue:", len(queue), "rows |", queue["client_id"].nunique(), "clients")
print("Priority tiers:", queue["priority_tier"].value_counts().to_dict())
print()
print("Primary reason-code counts:")
for code_name, n in queue["reason_code"].value_counts().items():
    print(f"  {code_name:28s} {n}")
print()
print("Archetype counts:", queue["archetype"].value_counts().to_dict())
print("Rows requiring human review:", int(queue["human_review_required"].sum()))

# --- Descriptive decay / refresh analysis (whole dataset, NOT a model claim) ---
order = {"0-30": 0, "31-90": 1, "91-180": 2, "181+": 3}
decay = df.groupby("freshness_tier")["trend_direction"].apply(lambda s: (s == "down").mean())
counts = df.groupby("freshness_tier")["trend_direction"].count()
decay_table = pd.DataFrame({"n": counts, "decline_rate": decay.round(4)}).loc[
    sorted(order, key=lambda k: order[k])
]
print()
print("Observed decline rate by days-since-last-update tier (descriptive, all 30,000 rows):")
print(decay_table)

Train fold : 23837 rows / 25 clients
Test fold  : 6163 rows / 7 clients
Client overlap: 0
Test base rate ('down' label): 0.511



Queue: 6163 rows | 7 clients
Priority tiers: {'P3_MONITOR': 2426, 'P2_MEDIUM': 1786, 'P4_LOW': 1579, 'P1_HIGH': 372}

Primary reason-code counts:
  RC02_CTR_OPPORTUNITY         2063
  RC05_MODEL_ONLY_WARNING      935
  RC07_LOW_PRIORITY            836
  RC01_HIGH_VALUE_DECLINE      685
  RC06_MONITOR                 639
  RC04_DEEP_POSITION_RISK      590
  RC03_AGING_CONTENT           415

Archetype counts: {'B_CTR_OPPORTUNITY': 2063, 'E_MODEL_ONLY_WARNING': 935, 'G_LOW_PRIORITY': 836, 'A_HIGH_VALUE_DECLINE': 685, 'F_MONITOR': 639, 'D_DEEP_POSITION_RISK': 590, 'C_AGING_CONTENT': 415}
Rows requiring human review: 4688

Observed decline rate by days-since-last-update tier (descriptive, all 30,000 rows):
                    n  decline_rate
freshness_tier                     
0-30            20480        0.5114
31-90             175        0.5886
91-180           9171        0.6111
181+              174        0.4713


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** The playbook is a decision-support tool that:
* prioritizes pages for **human review** (which page to look at first),
* identifies potentially declining content and useful refresh-review candidates,
* gives a transparent reason code, archetype, and high-level recommended action per page,
* supports content-team workload prioritization,
* and provides evidence for the research paper.

**Limitations — stated plainly.**

* **Predictions are probabilistic and the model has errors.** Out-of-sample precision here is measured, not assumed; every metric shown is recomputed in this notebook from the held-out fold.
* **No causality.** This is cross-sectional data — an observed association between, say, content age and decline does not mean updating content prevents decline. Nothing here says "refresh this page and traffic will improve."
* **Confounders exist.** Declining pages differ from healthy pages in many ways at once; the model cannot separate cause from correlation.
* **External factors.** Search performance changes for reasons outside this dataset (seasonality, competition, ranking-system changes).
* **Historical data ≠ future behavior.** The grouped split is one split of one snapshot; it is a measured indication of cross-client behavior, not a guarantee.
* **The model has no editorial or business context.** It sees no topic quality, intent freshness, or competitive dynamics.
* **Thresholds are decision-policy choices.** The priority tiers and the 0.5 decision boundary are proposed policy, not empirical truths.
* **Refresh impact is not measured.** This dataset has no intervention/outcome data, so the playbook cannot claim that a refresh produced an effect.

**Non-production scope.** This is a research/prototype decision-support system, **not** an autonomous production content-management system. It is not ready to automatically modify a website.

**Model performance vs business impact.** Model performance is a measured property of the classifier (Section 2 code). Business impact is what a refresh *actually* changes — which this dataset cannot measure. The two must never be conflated.

In [2]:
# Out-of-sample metrics on the exact fold that becomes the queue
auc = roc_auc_score(y_test, prob)
metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
    "auc": auc,
}
print("Out-of-sample metrics on the held-out fold (the queue's rows):")
for k, v in metrics.items():
    print(f"  {k:10s} {v:.4f}")
print("Confusion matrix (rows: true, cols: predicted):")
print(confusion_matrix(y_test, pred))

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    top = np.argsort(np.asarray(scores))[::-1][:k]
    return float(np.mean(y_true[top]))

print()
print("Precision at the top of the queue (fraction of top-K that truly declined):")
for k in (10, 25, 50, 100, 200):
    print(f"  precision@{k:<4d} {precision_at_k(y_test, prob, k):.3f}")

# Honest, measured context for the reviewer (not assumptions)
n_flags = int((queue["predicted_declining"]).sum())
n_false_alarms = int(((queue["y_true"] == 0) & (queue["predicted_declining"])).sum())
print()
print("Predicted-decline flags in the queue:", n_flags, "of", len(queue))
print("Measured false alarms among those flags (true label NOT declining):",
      n_false_alarms, f"({n_false_alarms / n_flags:.1%} of flags)")
print("Measured precision@50 (top 50 rows):", round(precision_at_k(y_test, prob, 50), 3))
print("Measured overall precision:", round(metrics["precision"], 3))
print("Test base rate for reference:", round(float(y_test.mean()), 4))

Out-of-sample metrics on the held-out fold (the queue's rows):
  accuracy   0.8142
  precision  0.8049
  recall     0.8399
  f1         0.8221
  auc        0.9054
Confusion matrix (rows: true, cols: predicted):
[[2373  641]
 [ 504 2645]]

Precision at the top of the queue (fraction of top-K that truly declined):
  precision@10   1.000
  precision@25   1.000
  precision@50   1.000
  precision@100  0.990
  precision@200  0.995

Predicted-decline flags in the queue: 3325 of 6163
Measured false alarms among those flags (true label NOT declining): 665 (20.0% of flags)
Measured precision@50 (top 50 rows): 1.0
Measured overall precision: 0.805
Test base rate for reference: 0.511


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human-review rules.**

| Situation | Rule |
| --- | --- |
| High model risk + strong business signal (`RC01`, `RC02`, `RC03` on high traffic) | **Review first** |
| High model risk + weak supporting evidence (`RC05_MODEL_ONLY_WARNING`) | **Manual investigation required** |
| Conflicting signals (e.g. strong-looking profile still predicted declining) | **Human review required** |
| Low-risk pages (`RC07`) | Monitor; do not act now |
| High-value / important pages | **Mandatory human review** |
| Predictions near the decision boundary (probability 0.5–0.65) | Do not act automatically; verify the trend by hand |
| Any irreversible change | **Human approval required** |

The decision chain is always:

`Prediction → recommendation → human validation → action`

never `Prediction → automatic action`.

**What a reviewer checks before acting on a flagged page.**

1. Re-derive the trend by hand from `impressions_last_30d` vs `impressions_prev_30d`. The label uses a ~20% drop threshold; a page near that boundary may be flagged even when the change is mild.
2. Check position data honestly. `avg_position == 0` means **no data** in this dataset (1,205 rows dataset-wide), never "rank zero" — such a page cannot be judged on visibility.
3. Check staleness and traffic together: "stale" alone is not "declining."
4. Add editorial context: topic quality, search intent, and competitive changes the model cannot see.

## What should NOT be automated

The system must **never** automatically:
* delete pages,
* publish rewritten content,
* modify production content,
* create redirects,
* change canonical URLs,
* change indexing directives,
* change SEO metadata without review,
* choose search intent or target keywords on its own,
* make irreversible website changes,
* make decisions for sensitive/high-stakes content,
* or retrain the model after every prediction.

Why: measured precision on this snapshot is below 1.0 (see Section 2) and the dataset establishes no causal or business-outcome evidence. Any of the actions above is a website change with consequences a flag alone cannot justify. Automated decisions would also bypass the editorial context the model explicitly lacks.

Section 3 code: show the readable **Top 25 queue**, and quantify the rows a reviewer must treat carefully (false alarms among flags, near-boundary predictions, and review-tier rows with missing position data).

In [3]:
from IPython.display import display

display_cols = [
    "rank", "content_id", "client_id", "priority_score", "priority_tier",
    "decline_probability", "predicted_declining", "reason_code", "archetype",
    "recommended_action", "human_review_required", "impressions_90d", "ctr",
    "avg_position", "days_since_last_update",
]
print("Top 25 action queue (held-out fold, out-of-sample scores):")
display(queue.head(25)[display_cols])

# Quantities a reviewer must treat carefully
n_flags = int((queue["predicted_declining"]).sum())
n_false_alarms = int(((queue["y_true"] == 0) & (queue["predicted_declining"])).sum())
n_near_boundary = int(((queue["decline_probability"] >= 0.5) & (queue["decline_probability"] < 0.65)).sum())
n_no_pos_review = int(((queue["priority_tier"].isin(["P1_HIGH", "P2_MEDIUM"])) & (queue["avg_position"] == 0)).sum())

print("Predicted-decline flags in the queue:", n_flags, "of", len(queue))
print("Measured false alarms among those flags (true label NOT declining):", n_false_alarms,
      f"({n_false_alarms / n_flags:.1%})")
print("Predicted-decline rows within 0.15 of the 0.5 decision boundary:", n_near_boundary)
print("Review-tier (P1/P2) rows with NO position data (avg_position == 0):", n_no_pos_review)

Top 25 action queue (held-out fold, out-of-sample scores):


,rank,content_id,client_id,priority_score,priority_tier,decline_probability,predicted_declining,reason_code,archetype,recommended_action,human_review_required,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_66458ac1b739,client_8527a891e2,84.9,P1_HIGH,0.930,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,6822,0.03,2.9,102
1,2,content_f55fd2d8ed04,client_4e07408562,83.7,P1_HIGH,0.980,True,RC02_CTR_OPPORTUNITY,B_CTR_OPPORTUNITY,"Review title, meta description, SERP alignment...",True,2237,0.09,1.3,104
2,3,content_ccf887ee3581,client_4e07408562,82.7,P1_HIGH,0.940,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,2993,0.03,5.5,104
3,4,content_3395aba722a2,client_8527a891e2,82.7,P1_HIGH,0.935,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,3320,0.09,11.1,102
4,5,content_907688f36763,client_f369cb89fc,82.6,P1_HIGH,0.905,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,11442,0.24,4.8,20
5,6,content_9ac61c04930e,client_8527a891e2,82.5,P1_HIGH,0.975,True,RC02_CTR_OPPORTUNITY,B_CTR_OPPORTUNITY,"Review title, meta description, SERP alignment...",True,1828,0.22,5.6,104
6,7,content_da33a53074bd,client_4e07408562,82.4,P1_HIGH,0.945,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,2678,0.04,5.7,104
7,8,content_6ac3ab740bbf,client_f369cb89fc,82.2,P1_HIGH,0.840,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,22462,0.14,4.6,106
8,9,content_e988c1699454,client_8527a891e2,81.9,P1_HIGH,0.950,True,RC02_CTR_OPPORTUNITY,B_CTR_OPPORTUNITY,"Review title, meta description, SERP alignment...",True,2197,0.00,21.5,104
9,10,content_454ad2864776,client_f369cb89fc,81.4,P1_HIGH,0.835,True,RC01_HIGH_VALUE_DECLINE,A_HIGH_VALUE_DECLINE,Prioritize human content-refresh review.,True,16190,0.29,23.9,106


Predicted-decline flags in the queue: 3325 of 6163
Measured false alarms among those flags (true label NOT declining): 665 (20.0%)
Predicted-decline rows within 0.15 of the 0.5 decision boundary: 1204
Review-tier (P1/P2) rows with NO position data (avg_position == 0): 0


## 4. Decay insight + monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Decay / refresh insight.** The relationship between time-since-last-update and decline is analyzed as *observed* data (Section 1 code): decline rate by `freshness_tier` over all 30,000 rows, with sample size per bucket. The result is reported exactly as measured — if the pattern is weak or non-monotonic, that is the finding. No causal claim is made; the language is "the observed data suggests / the analysis indicates an association."

**Monitoring framework (research/prototype policy, not production MLOps).**
* Data quality: missingness, unexpected feature ranges, row counts, categorical/value changes.
* Feature distribution drift: impressions, CTR, position, days-since-update, and other important model features.
* Model performance: compare against the validated grouped-split metrics computed in this notebook (Section 2) and committed in `work/outputs/action_playbook_metrics.json`.
* Label behavior: track the observed decline rate (base rate).

**Retrain triggers (proposed monitoring-policy thresholds — not empirically proven).**
* Trigger 1 — performance degradation: investigate a re-evaluation if precision@50 falls materially below the measured value here (guard band defined in code below).
* Trigger 2 — feature/data drift: investigate when important feature distributions change materially.
* Trigger 3 — label behavior: investigate if the observed decline rate changes substantially (e.g. ±0.05).
* Trigger 4 — data-pipeline changes: revalidate if feature definitions or data-generation logic change.
* Trigger 5 — business/process changes: revalidate if the definition of "declining" or the content workflow changes.

**Cost / value thinking.** High review priority corresponds to high business exposure (traffic), meaningful decline risk, and strong supporting evidence — a prioritization heuristic, **not** measured financial ROI. No claim like "refreshing this page will generate X dollars" is made anywhere in this project, because no revenue or intervention data exists.

Section 4 code: print the current reference values each trigger compares against, so the checkpoints have concrete numbers.

In [4]:
# Reference values for the monitoring triggers (current snapshot)
trigger_ref = {
    "snapshot_rows": int(len(df)),
    "scored_rows": int(len(queue)),
    "scored_clients": int(queue["client_id"].nunique()),
    "test_base_rate": round(float(y_test.mean()), 4),
    "precision_at_50": round(precision_at_k(y_test, prob, 50), 4),
    "overall_precision": round(metrics["precision"], 4),
    "predicted_decline_share": round(float(queue["predicted_declining"].mean()), 4),
    "share_aging_90d": round(float((queue["days_since_last_update"] >= 90).mean()), 4),
    "share_no_position_data": round(float((queue["avg_position"] == 0).mean()), 4),
}
for k, v in trigger_ref.items():
    print(f"  {k:24s} {v}")

print()
print("Proposed monitoring-policy thresholds (labeled policy, not empirically derived):")
print(f"  - Retrain / re-evaluate if precision@50 falls below {trigger_ref['precision_at_50'] - 0.10:.2f}")
print("    (measured precision@50 minus a 0.10 guard band)")
print("  - Re-pull data if the snapshot is older than 90 days")
print("  - Investigate if the observed decline rate moves by more than +-0.05 from the current base rate")
print("  - Revalidate the model if feature definitions or the content workflow change")

  snapshot_rows            30000
  scored_rows              6163
  scored_clients           7
  test_base_rate           0.511
  precision_at_50          1.0
  overall_precision        0.8049
  predicted_decline_share  0.5395
  share_aging_90d          0.1976
  share_no_position_data   0.0101

Proposed monitoring-policy thresholds (labeled policy, not empirically derived):
  - Retrain / re-evaluate if precision@50 falls below 0.90
    (measured precision@50 minus a 0.10 guard band)
  - Re-pull data if the snapshot is older than 90 days
  - Investigate if the observed decline rate moves by more than +-0.05 from the current base rate
  - Revalidate the model if feature definitions or the content workflow change


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

* `work/outputs/w07_ranked_action_queue.csv` — the ranked queue (held-out fold only), one row per content item, with priority, reason code, reason text, archetype, recommended action, and the key review columns. The queue CSV is **regenerated by this notebook and kept out of git** (the repo's CI leak-guard ignores `work/**/*.csv`).
* `work/outputs/action_summary.csv` — long-format table of measured metrics, tier sizes, reason-code counts, and archetype counts, ready for a paper table.
* `work/outputs/action_playbook_metrics.json` — the same numbers machine-readable; **committed** as the audit trail.
* `work/figures/` — four reusable paper figures:
  1. `w07_content_age_decline.png` — observed decline rate by freshness tier (Figure 1).
  2. `w07_action_archetypes.png` — distribution of recommended actions / archetypes (Figure 2).
  3. `w07_priority_tiers.png` — priority distribution (Figure 3).
  4. `w07_precision_at_k.png` — model risk: precision at the top of the queue (Figure 4).

In [5]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = os.path.join(repo_root(), "work", "outputs")
FIG = os.path.join(repo_root(), "work", "figures")
os.makedirs(OUT, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# --- 1. Ranked queue csv (gitignored, local only) ---
export_cols = [
    "rank", "content_id", "client_id", "decline_probability", "predicted_declining",
    "priority_score", "priority_tier", "reason_code", "reason_text",
    "reason_codes", "evidence_count", "archetype", "recommended_action",
    "human_review_required", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "days_since_last_update", "content_age_days", "word_count",
    "trend_direction", "freshness_tier", "impression_tier", "position_tier",
]
action_queue = queue.copy()
action_queue["reason_codes"] = action_queue["reason_codes"].apply(lambda r: "|".join(r))
queue_path = os.path.join(OUT, "w07_ranked_action_queue.csv")
action_queue[export_cols].to_csv(queue_path, index=False)

# --- 2. Metrics JSON (committed as the audit trail) ---
tier_counts = action_queue["priority_tier"].value_counts().to_dict()
reason_code_counts = action_queue["reason_code"].value_counts().to_dict()
archetype_counts = action_queue["archetype"].value_counts().to_dict()

metrics_payload = {
    "notebook": "w07_action_playbook",
    "playbook": "content_action_playbook",
    "target": "is_declining = (trend_direction == 'down')",
    "split": "grouped-by-client, GroupShuffleSplit(test_size=0.2, random_state=42)",
    "model": "RandomForestClassifier(n_estimators=200, random_state=42)",
    "train_fold": {"rows": int(len(train_idx)), "clients": int(len(train_clients))},
    "scored_fold": {"rows": int(len(action_queue)), "clients": int(action_queue["client_id"].nunique())},
    "test_base_rate": round(float(y_test.mean()), 4),
    "metrics": {k: round(float(v), 4) for k, v in metrics.items()},
    "precision_at_k": {
        f"precision_at_{k}": round(precision_at_k(y_test, prob, k), 4)
        for k in (10, 25, 50, 100, 200)
    },
    "false_alarm_share_of_flags": round(n_false_alarms / n_flags, 4),
    "priority_tiers": {k: int(v) for k, v in tier_counts.items()},
    "primary_reason_codes": {k: int(v) for k, v in reason_code_counts.items()},
    "archetypes": {k: int(v) for k, v in archetype_counts.items()},
    "observed_decay_by_freshness_tier": {
        tier: {"n": int(decay_table.loc[tier, "n"]), "decline_rate": float(decay_table.loc[tier, "decline_rate"])}
        for tier in order
    },
    "monitoring_reference": trigger_ref,
    "note": "priority_score is a prioritization heuristic (0.6*risk + 0.3*value + 0.1*staleness), not revenue; tier thresholds are proposed decision-policy choices.",
}
metrics_path = os.path.join(OUT, "action_playbook_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as fh:
    json.dump(metrics_payload, fh, indent=2, sort_keys=True)

# --- 3. Summary csv (long format, gitignored) ---
summary_rows = []
for k, v in metrics.items():
    summary_rows.append({"metric": k, "value": round(float(v), 4), "note": "held-out fold"})
for k, v in tier_counts.items():
    summary_rows.append({"metric": f"tier_{k}", "value": int(v), "note": "queue size"})
for k, v in reason_code_counts.items():
    summary_rows.append({"metric": f"reason_{k}", "value": int(v), "note": "queue rows"})
for k, v in archetype_counts.items():
    summary_rows.append({"metric": f"archetype_{k}", "value": int(v), "note": "queue rows"})
summary_path = os.path.join(OUT, "action_summary.csv")
pd.DataFrame(summary_rows).to_csv(summary_path, index=False)

# --- 4. Figures ---
def save_bar(fig_path, labels, values, title, color="#6F4E7C"):
    fig, ax = plt.subplots(figsize=(6.5, 3.8), dpi=150)
    ax.bar(labels, values, color=color)
    ax.set_title(title, fontsize=11)
    ax.tick_params(labelsize=8)
    for i, v in enumerate(values):
        ax.text(i, v, f" {v:,.0f}", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(fig_path)
    plt.close(fig)

# Figure 1: content age vs observed decline rate
tiers = list(order)
fig_path1 = os.path.join(FIG, "w07_content_age_decline.png")
fig, ax = plt.subplots(figsize=(6.5, 3.8), dpi=150)
ax.bar(tiers, [decay_table.loc[t, "decline_rate"] for t in tiers], color="#7C6F4E")
ax.set_title("Observed decline rate by freshness tier (all 30,000 rows)", fontsize=11)
ax.set_ylabel("share labeled declining", fontsize=9)
ax.set_ylim(0, 0.75)
ax.tick_params(labelsize=8)
for i, t in enumerate(tiers):
    ax.text(i, decay_table.loc[t, "decline_rate"] + 0.01,
            f"n={int(decay_table.loc[t, 'n']):,}", ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(fig_path1)
plt.close(fig)

# Figure 2: action / archetype distribution
archetype_order = sorted(archetype_counts, key=lambda a: a.split("_")[0])
fig_path2 = os.path.join(FIG, "w07_action_archetypes.png")
labels2 = [a.split("_", 1)[1] for a in archetype_order]
save_bar(fig_path2, labels2, [archetype_counts[a] for a in archetype_order],
         "Recommended-action / archetype distribution (held-out queue)", color="#4E7C6F")

# Figure 3: priority distribution
tier_order = ["P1_HIGH", "P2_MEDIUM", "P3_MONITOR", "P4_LOW"]
fig_path3 = os.path.join(FIG, "w07_priority_tiers.png")
save_bar(fig_path3, tier_order, [tier_counts.get(t, 0) for t in tier_order],
         "Queue size by priority tier", color="#4E6F7C")

# Figure 4: precision at top of queue (model risk)
fig_path4 = os.path.join(FIG, "w07_precision_at_k.png")
fig, ax = plt.subplots(figsize=(6.5, 3.8), dpi=150)
ks = list(range(10, 501, 10))
pks = [precision_at_k(y_test, prob, k) for k in ks]
ax.plot(ks, pks, marker="o", markersize=2, linewidth=1.4, color="#6F4E7C")
ax.axhline(y_test.mean(), color="#8a2b2b", linestyle="--", linewidth=1)
ax.text(ks[-1], y_test.mean() + 0.01, "base rate", color="#8a2b2b", fontsize=8, ha="right")
ax.set_xlabel("K (top of queue)", fontsize=9)
ax.set_ylabel("Precision@K", fontsize=9)
ax.set_title("Precision at the top of the held-out queue", fontsize=11)
ax.set_ylim(0.3, 1.05)
fig.tight_layout()
fig.savefig(fig_path4)
plt.close(fig)

fig_files = ["w07_content_age_decline.png", "w07_action_archetypes.png",
             "w07_priority_tiers.png", "w07_precision_at_k.png"]

# --- 5. Verify what was written ---
print("Wrote outputs:")
for p in sorted(os.listdir(OUT)):
    full = os.path.join(OUT, p)
    print(f"  outputs/{p}  ({os.path.getsize(full):,} bytes)")
print("Wrote figures:")
for p in sorted(os.listdir(FIG)):
    full = os.path.join(FIG, p)
    print(f"  figures/{p}  ({os.path.getsize(full):,} bytes)")

recheck = pd.read_csv(queue_path)
assert len(recheck) == len(action_queue)
assert recheck["rank"].is_monotonic_increasing
print()
print("Verification: queue csv rows match in-memory queue; ranks are monotonic.")
print("Top of the queue:")
print(recheck.head(5)[["rank", "priority_tier", "decline_probability", "reason_code", "archetype"]].to_string(index=False))

Wrote outputs:
  outputs/action_playbook_metrics.json  (2,247 bytes)
  outputs/action_summary.csv  (931 bytes)
  outputs/w07_ranked_action_queue.csv  (2,479,427 bytes)
Wrote figures:
  figures/w07_action_archetypes.png  (41,823 bytes)
  figures/w07_content_age_decline.png  (33,914 bytes)
  figures/w07_precision_at_k.png  (32,844 bytes)
  figures/w07_priority_tiers.png  (24,226 bytes)

Verification: queue csv rows match in-memory queue; ranks are monotonic.
Top of the queue:
 rank priority_tier  decline_probability             reason_code            archetype
    1       P1_HIGH                0.930 RC01_HIGH_VALUE_DECLINE A_HIGH_VALUE_DECLINE
    2       P1_HIGH                0.980    RC02_CTR_OPPORTUNITY    B_CTR_OPPORTUNITY
    3       P1_HIGH                0.940 RC01_HIGH_VALUE_DECLINE A_HIGH_VALUE_DECLINE
    4       P1_HIGH                0.935 RC01_HIGH_VALUE_DECLINE A_HIGH_VALUE_DECLINE
    5       P1_HIGH                0.905 RC01_HIGH_VALUE_DECLINE A_HIGH_VALUE_DECLINE


## 6. Self-check

Final checklist for this notebook:

- [x] Ranked queue generated (held-out grouped-by-client fold only)
- [x] Reason codes + reason text generated (deterministic rules, RC01–RC07)
- [x] Archetypes assigned (A–G, deterministic mapping)
- [x] Recommended actions assigned
- [x] Human-review flag assigned per row
- [x] Intended use documented
- [x] Limitations documented
- [x] Human-review rules documented
- [x] No-go automation cases documented
- [x] Decay / refresh analysis completed (observed, no causal claim)
- [x] Monitoring triggers documented (proposed policy)
- [x] Retraining triggers documented (proposed policy)
- [x] Cost/value framework documented (heuristic, not ROI)
- [x] Queue exported to `work/outputs/w07_ranked_action_queue.csv`
- [x] Figures exported to `work/figures/`
- [x] Metrics JSON written (audit trail, committed)
- [x] Existing metrics JSONs untouched
- [x] Notebook executed successfully top to bottom

The code cell below runs automated checks against the exported queue and files.

In [6]:
# Automated self-checks (Section 6)
assert len(action_queue) > 0, "empty queue"

required_cols = [
    "rank", "content_id", "decline_probability", "predicted_declining",
    "priority_score", "priority_tier", "reason_code", "reason_text",
    "archetype", "recommended_action", "human_review_required",
]
for col in required_cols:
    assert col in action_queue.columns, f"missing column: {col}"

assert action_queue["reason_code"].notna().all()
assert action_queue["reason_text"].notna().all()
assert action_queue["archetype"].notna().all()
assert action_queue["recommended_action"].notna().all()
assert action_queue["human_review_required"].notna().all()
assert action_queue["rank"].is_monotonic_increasing
assert ((action_queue["decline_probability"] >= 0) & (action_queue["decline_probability"] <= 1)).all()
assert pd.to_numeric(action_queue["priority_score"], errors="coerce").notna().all()

assert os.path.exists(queue_path), "queue csv missing"
assert os.path.exists(metrics_path), "metrics json missing"
for f in fig_files:
    assert os.path.exists(os.path.join(FIG, f)), f"figure missing: {f}"

print("Automated self-checks passed: queue columns, non-null values, rank order,")
print("probability bounds, priority scores, and all exported files verified.")

Automated self-checks passed: queue columns, non-null values, rank order,
probability bounds, priority scores, and all exported files verified.
